# 05: Offline Recommender Evaluation, Ablation Study & Hybrid Demo
This notebook demonstrates full temporal offline evaluation, ablation study, and the unified RecommendationEngine with cold-start, sparse, and mature customer routing.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

# Robust root resolver whether run from notebooks/ or workspace root
_curr = Path('.').resolve()
PROJECT_ROOT = _curr if (_curr / 'src').exists() else (_curr.parent if (_curr.parent / 'src').exists() else (_curr.parents[1] if len(_curr.parents) > 1 and (_curr.parents[1] / 'src').exists() else _curr))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Switch working directory to project root so all relative paths ('data/...', 'artifacts/...') resolve seamlessly
os.chdir(PROJECT_ROOT)

from src.recommenders.hybrid import RecommendationEngine


## 1. Load Production Recommendation Engine & Evaluation Results

In [2]:
engine = RecommendationEngine.load('artifacts')
eval_df = pd.read_csv('outputs/metrics/evaluation_metrics.csv')
ablation_df = pd.read_csv('outputs/metrics/ablation_study.csv')
print("Offline Evaluation Summary:")
eval_df


c:\Users\hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Offline Evaluation Summary:


,model,eval_users,precision@5,recall@5,hit_rate@5,map@5,ndcg@5,precision@10,recall@10,hit_rate@10,map@10,ndcg@10,precision@20,recall@20,hit_rate@20,map@20,ndcg@20,diversity@10,novelty@10,catalog_coverage
0,Popularity Baseline,1852,0.0649,0.1013,0.2721,0.0796,0.1136,0.0425,0.1280,0.3278,0.0730,0.1129,0.0252,0.1500,0.3661,0.0736,0.1185,0.8971,16.6096,0.0362
1,Trending Baseline,1852,0.0645,0.1014,0.2711,0.0796,0.1134,0.0424,0.1270,0.3261,0.0730,0.1127,0.0258,0.1535,0.3742,0.0739,0.1197,0.8961,16.6096,0.0362
2,Item-Item Collaborative Filtering,1852,0.0501,0.0737,0.2230,0.0545,0.0819,0.0397,0.1134,0.3105,0.0532,0.0902,0.0335,0.1881,0.4336,0.0589,0.1144,0.5189,16.6007,0.9975
3,ALS Matrix Factorization,1852,0.0255,0.0328,0.1134,0.0198,0.0336,0.0254,0.0671,0.2009,0.0214,0.0445,0.0244,0.1308,0.3299,0.0264,0.0656,0.4045,16.6007,1.0000
4,TruncatedSVD Baseline,1852,0.0243,0.0303,0.1112,0.0172,0.0307,0.0239,0.0593,0.1928,0.0185,0.0400,0.0231,0.1167,0.3072,0.0232,0.0596,0.3081,16.6007,0.8087
5,Hybrid Engine,1852,0.0532,0.0761,0.2311,0.0646,0.0930,0.0398,0.1160,0.3110,0.0618,0.0993,0.0335,0.1895,0.4401,0.0672,0.1231,0.5856,16.6096,1.0000
6,Hybrid (Sparse Cohort),88,0.0273,0.0909,0.1364,0.0441,0.0614,0.0239,0.1629,0.2273,0.0549,0.0875,0.0148,0.2027,0.2614,0.0586,0.0986,0.6035,16.6096,0.7712
7,Hybrid (Mature Cohort),1764,0.0545,0.0753,0.2358,0.0657,0.0946,0.0406,0.1137,0.3152,0.0621,0.0999,0.0345,0.1888,0.4490,0.0677,0.1243,0.5847,16.6096,1.0000


## 2. Component Ablation Study Analysis

In [3]:
print("Ablation Study Results:")
ablation_df[['model', 'recall@10', 'recall@20', 'hit_rate@10', 'hit_rate@20', 'ndcg@10', 'catalog_coverage']]


Ablation Study Results:


,model,recall@10,recall@20,hit_rate@10,hit_rate@20,ndcg@10,catalog_coverage
0,Popularity Only,0.1280,0.1500,0.3278,0.3661,0.1129,0.0362
1,Popularity + Segment,0.1291,0.1525,0.3272,0.3720,0.1131,0.0550
2,Item-Item CF Only,0.1134,0.1881,0.3105,0.4336,0.0902,0.9975
3,ALS Matrix Factorization Only,0.0671,0.1308,0.2009,0.3299,0.0445,1.0000
4,ALS + Item-Item,0.0852,0.1736,0.2468,0.4125,0.0524,1.0000
5,ALS + Segment,0.0769,0.1606,0.2235,0.3877,0.0496,1.0000
6,ALS + Item-Item + Segment,0.1101,0.1939,0.3029,0.4411,0.0779,1.0000
7,Full Hybrid Engine,0.1250,0.2024,0.3321,0.4530,0.1041,1.0000


## 3. Live Recommendation Engine Demo across Customer States

In [4]:
# Case A: Brand New Customer (Cold-Start)
cold_res = engine.recommend(customer_id=9999999, n=5)
print(f"Cold-Start User Strategy: {cold_res['recommendation_type']}")
for r in cold_res['recommendations']:
    print(f"  Rank #{r['rank']} Product #{r['product_id']} (Score: {r['score']:.3f}, Sources: {r['sources']}): {r['explanation']}")

# Case B: Mature Customer (Full Hybrid)
mature_user = list(engine.user_interaction_counts.keys())[0]
mature_res = engine.recommend(customer_id=mature_user, n=5)
print(f"\nMature User Strategy ({mature_res['segment_label']}): {mature_res['recommendation_type']}")
for r in mature_res['recommendations']:
    print(f"  Rank #{r['rank']} Product #{r['product_id']} (Score: {r['score']:.3f}, Sources: {r['sources']}): {r['explanation']}")


Cold-Start User Strategy: cold_start_user
  Rank #1 Product #1000 (Score: 1.000, Sources: ['popularity', 'trending']): Trending with high momentum across the store right now.
  Rank #2 Product #1001 (Score: 0.403, Sources: ['popularity', 'trending']): Trending with high momentum across the store right now.
  Rank #3 Product #1002 (Score: 0.248, Sources: ['popularity', 'trending']): Trending with high momentum across the store right now.
  Rank #4 Product #1003 (Score: 0.149, Sources: ['popularity', 'trending']): Trending with high momentum across the store right now.
  Rank #5 Product #1004 (Score: 0.098, Sources: ['popularity', 'trending']): Trending with high momentum across the store right now.

Mature User Strategy (High-Value Frequent Active Shoppers): hybrid
  Rank #1 Product #1000 (Score: 0.732, Sources: ['als', 'segment', 'popularity', 'trending']): Popular among customers in your segment (High-Value Frequent Active Shoppers).
  Rank #2 Product #1002 (Score: 0.426, Sources: ['a